# C3 — Baseline e Regressão Logística

## Objetivo

Rodar o `DummyClassifier` (baseline) e a `LogisticRegression` sobre a partição persistida pelo Notebook 03, usando **somente treino e validação**.

Este notebook:

- carrega o artefato supervisionado canônico via `load_supervised_artifact`;
- valida o contrato do manifesto (partições, alvo, features);
- **não abre o teste**;
- não faz tuning, calibração nem escolha de limiar;
- registra métricas e predições de validação em `outputs/`.

> O limiar usado aqui (0.5) é apenas diagnóstico. A escolha definitiva pertence ao C5.

## 1. Bootstrap — raiz do projeto e imports

Toda a configuração vem de `src.utils.config` e `configs/supervised.yaml`. Nenhum caminho absoluto, nenhuma feature hard-coded.

In [ ]:
from pathlib import Path
import os
import sys
import platform

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src").is_dir() and (pasta / "configs").is_dir():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("ROOT_DIR:", ROOT_DIR)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

In [ ]:
from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import load_supervised_artifact
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas
from src.models.baseline import build_baseline_pipeline, evaluate_baseline
from src.models.logistic import build_logistic_pipeline

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print(C.resumo())

## 2. Carregar o artefato supervisionado canônico

A partição já foi criada e persistida pelo Notebook 03. Este notebook **reutiliza** a partição, não cria outra.

O tipo de coorte vem de `SRAG_COHORT_KIND` (padrão `provisional`), alinhado com a decisão de manter explícita a fonte durante o desenvolvimento.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)

print("Tipo de coorte:", manifest["cohort_kind"])
print("Dataset:", manifest["dataset_path"])
print("SHA-256:", manifest["dataset_sha256"])
print("Linhas:", manifest["rows"])
print("Colunas:", manifest["columns"])
print("Seed:", manifest["seed"])
print("Teste aberto?", manifest["test_opened"])
print("Partições:", manifest["partition_counts"])

display(df.head(3))

## 3. Validar o contrato do manifesto

Antes de qualquer modelagem, confirmamos:

- `partition_column`, `target_column` e `feature_columns` existem no dataframe;
- as três partições estão presentes (`treino`, `validacao`, `teste`);
- o teste **não** foi aberto;
- o alvo é binário 0/1.

In [ ]:
PARTICAO = manifest["partition_column"]
ALVO = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

particoes_observadas = set(
    df[PARTICAO].dropna().astype(str).unique().tolist()
)

alvo_binario = (
    pd.to_numeric(df[ALVO], errors="coerce")
    .dropna()
    .astype(int)
    .isin([0, 1])
    .all()
)

verificacoes = {
    "artefato_nao_vazio": not df.empty,
    "particao_presente": PARTICAO in df.columns,
    "alvo_presente": ALVO in df.columns,
    "features_presentes": set(FEATURES).issubset(df.columns),
    "particoes_esperadas": particoes_observadas == {"treino", "validacao", "teste"},
    "alvo_binario": bool(alvo_binario),
    "teste_fechado_no_manifesto": manifest.get("test_opened") is False,
}

contrato_df = pd.DataFrame(
    {"verificacao": list(verificacoes.keys()), "resultado": list(verificacoes.values())}
)

display(contrato_df)

if not all(verificacoes.values()):
    falhas = [k for k, v in verificacoes.items() if not v]
    raise RuntimeError(f"Falhas no contrato do artefato: {falhas}")

print("Contrato validado. Nenhuma partição nova foi criada.")
print("Coluna de partição:", PARTICAO)
print("Coluna alvo:", ALVO)
print("Quantidade de features:", len(FEATURES))

## 4. Separar treino e validação (sem tocar no teste)

Somente `treino` e `validacao` são materializados. O bloco `teste` permanece no dataframe, mas não é selecionado em nenhuma célula deste notebook.

In [ ]:
treino = df.loc[df[PARTICAO] == "treino"].copy()
validacao = df.loc[df[PARTICAO] == "validacao"].copy()

# Não criamos a variável `teste`. Ela permanece intocada em `df`.

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")

X_val = validacao[FEATURES].copy()
y_val = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)

resumo_particoes = pd.DataFrame(
    {
        "particao": ["treino", "validacao"],
        "n": [len(treino), len(validacao)],
        "positivos": [int(y_train.sum()), int(y_val.sum())],
        "prevalencia": [float(y_train.mean()), float(y_val.mean())],
    }
)

display(resumo_particoes)
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val  :", X_val.shape, "| y_val  :", y_val.shape)

n_teste = int((df[PARTICAO] == "teste").sum())
print(f"Teste: {n_teste:,} linhas — mantido fechado.")

## 5. Checagem de leakage

`check_leakage` lê `features.proibidas` do `supervised.yaml` e confirma que nenhuma delas está em `X`.

In [ ]:
check_leakage(X_train)
check_leakage(X_val)

proibidas = set(C.cfg("features", "proibidas") or [])
assert not (set(X_train.columns) & proibidas)
assert not (set(X_val.columns) & proibidas)
print("Nenhuma coluna proibida presente em X_train ou X_val.")

## 6. Baseline — `DummyClassifier`

Estratégia `prior` (default do YAML): prevê a prevalência do treino. É o piso mínimo que qualquer modelo real precisa superar.

In [ ]:
estrategia_dummy = (
    C.cfg("models", "baseline", "strategy") or "prior"
)

metricas_dummy = evaluate_baseline(
    X_train, y_train, X_val, y_val, strategy=estrategia_dummy
)

display(pd.DataFrame([metricas_dummy]))

## 7. Regressão Logística

`C=1.0` e `lbfgs` são apenas um ponto de partida — nenhuma decisão de tuning é tomada aqui. O `class_weight` vem do YAML.

In [ ]:
log_cfg = C.cfg("models", "logistic") or {}

pipe_log = build_logistic_pipeline(
    C=1.0,
    solver=str(log_cfg.get("solver", "lbfgs")),
    max_iter=int(log_cfg.get("max_iter", 1000)),
    class_weight=log_cfg.get("class_weight", "balanced"),
)

pipe_log.fit(X_train, y_train)
prob_val = pipe_log.predict_proba(X_val)[:, 1]

if not np.isfinite(prob_val).all():
    raise RuntimeError("Probabilidades não finitas na validação.")
if not ((prob_val >= 0) & (prob_val <= 1)).all():
    raise RuntimeError("Probabilidades fora de [0, 1] na validação.")

metricas_log = metricas_completas(y_val, prob_val, limiar=0.5)
metricas_log["modelo"] = "logistica_C1"

display(pd.DataFrame([metricas_log]))

## 8. Comparação na validação

Comparação provisória entre o piso (`Dummy`) e a logística. Nenhum dos dois é finalista — essa decisão pertence ao C4/C5.

In [ ]:
linha_dummy = {
    "modelo": "dummy_prior",
    "auc": metricas_dummy["auc"],
    "ap": metricas_dummy["ap"],
    "brier": metricas_dummy["brier"],
    "f1": metricas_dummy["f1"],
    "precision": metricas_dummy["precision"],
    "recall": metricas_dummy["recall"],
    "accuracy": metricas_dummy["accuracy"],
    "limiar_diagnostico": metricas_dummy["limiar"],
    "n_validacao": int(metricas_dummy["n"]),
}

linha_log = {
    "modelo": "logistica_C1",
    "auc": metricas_log["auc"],
    "ap": metricas_log["ap"],
    "brier": metricas_log["brier"],
    "f1": metricas_log["f1"],
    "precision": metricas_log["precision"],
    "recall": metricas_log["recall"],
    "accuracy": metricas_log["accuracy"],
    "limiar_diagnostico": metricas_log["limiar"],
    "n_validacao": int(metricas_log["n"]),
}

comparacao = pd.DataFrame([linha_dummy, linha_log])
comparacao = comparacao.sort_values("ap", ascending=False).reset_index(drop=True)

display(comparacao)

delta_ap = float(linha_log["ap"] - linha_dummy["ap"])
print(f"Δ AP (logística − dummy): {delta_ap:+.4f}")
print("Se Δ ≤ 0, o pipeline precisa ser revisto antes de avançar.")

## 9. Predições de validação

Construímos uma tabela de predições apenas para a validação. Nada do teste.

In [ ]:
pred_val = pd.DataFrame(
    {
        "y_true": y_val.to_numpy(),
        "y_prob": prob_val,
        "modelo": "logistica_C1",
        "cohort_kind": manifest["cohort_kind"],
    }
)

pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

display(pred_val.head())
print("Total de predições de validação:", len(pred_val))

assert len(pred_val) == len(y_val)
assert pred_val["y_prob"].between(0, 1).all()

## 10. Persistência dos artefatos

Salvamos tabelas em `outputs/tables/` e predições em `outputs/predictions/`. Nada de modelo treinado como candidato final.

In [ ]:
import json

timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo = f"C3_baseline_logistica_{manifest['cohort_kind']}_{timestamp}"

caminho_metricas = TABLES_DIR / f"{prefixo}_metricas.csv"
caminho_comparacao = TABLES_DIR / f"{prefixo}_comparacao.csv"
caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"

comparacao.to_csv(caminho_comparacao, index=False, encoding="utf-8-sig")
pred_val.to_parquet(caminho_predicoes, index=False)

manifesto_c3 = {
    "cohort_kind": manifest["cohort_kind"],
    "dataset_path": manifest["dataset_path"],
    "dataset_sha256": manifest["dataset_sha256"],
    "feature_columns": FEATURES,
    "target_column": ALVO,
    "partition_column": PARTICAO,
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "n_teste_fechado": n_teste,
    "limiar_usado": 0.5,
    "limiar_e_diagnostico": True,
    "tuning_realizado": False,
    "calibracao_aplicada": False,
    "teste_aberto": False,
    "modelos": ["dummy_prior", "logistica_C1"],
    "artefatos": {
        "comparacao": str(caminho_comparacao),
        "predicoes_validacao": str(caminho_predicoes),
    },
}

caminho_manifesto.write_text(
    json.dumps(manifesto_c3, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

# Round-trip mínimo
comp_lido = pd.read_csv(caminho_comparacao, encoding="utf-8-sig")
pred_lido = pd.read_parquet(caminho_predicoes)

assert len(comp_lido) == len(comparacao)
assert len(pred_lido) == len(pred_val)
assert pred_lido["y_prob"].between(0, 1).all()

print("Comparação    :", caminho_comparacao)
print("Predições     :", caminho_predicoes)
print("Manifesto     :", caminho_manifesto)

## 11. Registro da execução

Registramos a execução no `Run` para reprodutibilidade. Nenhum resultado do teste entra aqui.

In [ ]:
with Run(
    modelo="C3_baseline_logistica",
    parametros={
        "cohort_kind": manifest["cohort_kind"],
        "estrategia_dummy": estrategia_dummy,
        "logistic_C": 1.0,
        "logistic_solver": log_cfg.get("solver", "lbfgs"),
        "limiar_diagnostico": 0.5,
        "tuning_realizado": False,
        "calibracao_aplicada": False,
        "teste_aberto": False,
    },
    dataset={
        "n_treino": int(len(treino)),
        "n_validacao": int(len(validacao)),
        "n_teste_fechado": n_teste,
    },
    features={
        "n_features": len(FEATURES),
        "usadas": FEATURES,
    },
    notas=(
        "C3 — baseline e logística na partição persistida. "
        "Limiar 0.5 apenas diagnóstico. Teste não consultado."
    ),
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(
        dummy_ap=float(metricas_dummy["ap"]),
        dummy_auc=float(metricas_dummy["auc"]),
        logistica_ap=float(metricas_log["ap"]),
        logistica_auc=float(metricas_log["auc"]),
        logistica_brier=float(metricas_log["brier"]),
    )

print("Run registrado em:", r.pasta)

## 12. Conclusão — baseline vs logística

Checagens de encerramento. O teste permaneceu fechado do começo ao fim.

In [ ]:
checklist = {
    "contrato_validado": all(verificacoes.values()),
    "teste_nao_materializado": "teste" not in globals(),
    "dummy_executado": "metricas_dummy" in globals(),
    "logistica_executada": "metricas_log" in globals(),
    "probabilidades_validas": bool(pred_val["y_prob"].between(0, 1).all()),
    "artefatos_reabertos": True,
    "limiar_nao_escolhido": True,
    "tuning_nao_realizado": True,
    "calibracao_nao_aplicada": True,
}

checklist_df = pd.DataFrame(
    {"item": list(checklist.keys()), "concluido": list(checklist.values())}
)
display(checklist_df)

if not all(checklist.values()):
    pendencias = [k for k, v in checklist.items() if not v]
    raise RuntimeError(f"C3 terminou com pendências: {pendencias}")

print("C3 concluído. Próximo: C4 — Boosting.")
print("Lembretes: teste fechado, limiar 0.5 apenas diagnóstico, nenhum tuning/calibração nesta etapa.")